# Diabetes 30-Day Readmission Prediction

Model-building workflow for predicting 30-day hospital readmission.

In [25]:
from pathlib import Path

import pandas as pd
import numpy as np

In [26]:
# Set project paths
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_FILE = DATA_DIR / "diabetic_data.csv"
MAPPING_FILE = DATA_DIR / "IDS_mapping.csv"

In [27]:
# Load raw data
data = pd.read_csv(DATA_FILE)

print("Shape:", data.shape)
data.head()

Shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Target Definition

The prediction target is 30-day hospital readmission.

- `1` = readmitted within 30 days (`<30`)
- `0` = not readmitted within 30 days (`>30` or `NO`)

In [28]:
# Create target
data["readmitted_30d"] = (data["readmitted"] == "<30").astype(int)

print(data["readmitted_30d"].value_counts())
print(data["readmitted_30d"].value_counts(normalize=True))

readmitted_30d
0    90409
1    11357
Name: count, dtype: int64
readmitted_30d
0    0.888401
1    0.111599
Name: proportion, dtype: float64


In [29]:
# Load ID mappings
mapping = pd.read_csv(MAPPING_FILE)

mapping.head()

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available


In [30]:
# Build lookup tables
admission_type_mapping = mapping.iloc[0:8].copy()

discharge_disposition_mapping = mapping.iloc[10:40].copy()
discharge_disposition_mapping.columns = [
    "discharge_disposition_id",
    "description"
]

admission_source_mapping = mapping.iloc[42:67].copy()
admission_source_mapping.columns = [
    "admission_source_id",
    "description"
]

print("Admission types:", len(admission_type_mapping))
print("Discharge dispositions:", len(discharge_disposition_mapping))
print("Admission sources:", len(admission_source_mapping))

Admission types: 8
Discharge dispositions: 30
Admission sources: 25


In [31]:
# Create lookup dictionaries
admission_type_lookup = dict(
    zip(
        admission_type_mapping["admission_type_id"].astype(str),
        admission_type_mapping["description"]
    )
)

discharge_disposition_lookup = dict(
    zip(
        discharge_disposition_mapping["discharge_disposition_id"].astype(str),
        discharge_disposition_mapping["description"]
    )
)

admission_source_lookup = dict(
    zip(
        admission_source_mapping["admission_source_id"].astype(str),
        admission_source_mapping["description"]
    )
)

In [32]:
# Decode ID variables
# Decode ID variables
data["admission_type"] = data["admission_type_id"].astype(str).map(
    admission_type_lookup
)

data["discharge_disposition"] = data["discharge_disposition_id"].astype(str).map(
    discharge_disposition_lookup
)

data["admission_source"] = data["admission_source_id"].astype(str).map(
    admission_source_lookup
)

print("Admission type missing:", data["admission_type"].isna().sum())
print("Admission source missing:", data["admission_source"].isna().sum())
print("Discharge disposition missing:", data["discharge_disposition"].isna().sum())

Admission type missing: 5291
Admission source missing: 6781
Discharge disposition missing: 3691


In [33]:
# Find unmapped IDs
print("Unmapped admission source IDs:")
print(
    sorted(
        set(data["admission_source_id"].astype(str))
        - set(admission_source_lookup.keys())
    )
)

print("\nUnmapped discharge disposition IDs:")
print(
    sorted(
        set(data["discharge_disposition_id"].astype(str))
        - set(discharge_disposition_lookup.keys())
    )
)

Unmapped admission source IDs:
[]

Unmapped discharge disposition IDs:
[]


In [34]:
print("Admission source IDs with missing descriptions:")
print(
    data.loc[
        data["admission_source"].isna(),
        "admission_source_id"
    ].value_counts()
)

print("\nDischarge disposition IDs with missing descriptions:")
print(
    data.loc[
        data["discharge_disposition"].isna(),
        "discharge_disposition_id"
    ].value_counts()
)

Admission source IDs with missing descriptions:
admission_source_id
17    6781
Name: count, dtype: int64

Discharge disposition IDs with missing descriptions:
discharge_disposition_id
18    3691
Name: count, dtype: int64


## Baseline Feature Set

The baseline uses information available by hospital discharge.

Identifiers and the target are excluded.

`weight` and `payer_code` are excluded based on the project feature-selection decisions.

The original diagnosis codes are retained rather than arbitrarily converting ICD-9 codes into numeric values.

In [35]:
# Define numeric features
numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

In [36]:
# Define categorical features
categorical_features = [
    "race",
    "gender",
    "age",
    "admission_type",
    "admission_source",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
    "change",
    "diabetesMed",
    "medical_specialty",
    "discharge_disposition",
]

In [37]:
# Build modeling matrix
candidate_features = numeric_features + categorical_features

X = data[candidate_features].copy()
y = data["readmitted_30d"].copy()

groups = data["patient_nbr"].copy()

print("Features:", X.shape)
print("Target:", y.shape)
print("Groups:", groups.shape)
print("Feature count:", len(candidate_features))

Features: (101766, 45)
Target: (101766,)
Groups: (101766,)
Feature count: 45


In [38]:
# Check feature columns
print("Missing from X:")
print(
    sorted(
        set(candidate_features) - set(X.columns)
    )
)

print("\nUnexpected columns:")
print(
    sorted(
        set(X.columns) - set(candidate_features)
    )
)

Missing from X:
[]

Unexpected columns:
[]


In [39]:
# Check target balance
target_counts = y.value_counts()
target_rates = y.value_counts(normalize=True)

print("Counts:")
print(target_counts)

print("\nProportions:")
print(target_rates)

Counts:
readmitted_30d
0    90409
1    11357
Name: count, dtype: int64

Proportions:
readmitted_30d
0    0.888401
1    0.111599
Name: proportion, dtype: float64


## Patient-Level Stratification

A patient can have multiple hospital encounters.

We therefore need to split at the patient level rather than randomly splitting individual encounters.

For stratification, each patient is assigned a patient-level target:

- `1` = the patient has at least one 30-day readmission in the dataset
- `0` = the patient has no 30-day readmission in the dataset

This keeps all encounters from the same patient together during splitting.

In [40]:
# Create patient-level target
patient_target = (
    data.groupby("patient_nbr")["readmitted_30d"]
    .max()
    .reset_index()
)

print(patient_target["readmitted_30d"].value_counts())

print("\nProportions:")
print(
    patient_target["readmitted_30d"].value_counts(
        normalize=True
    )
)

readmitted_30d
0    62684
1     8834
Name: count, dtype: int64

Proportions:
readmitted_30d
0    0.876479
1    0.123521
Name: proportion, dtype: float64


In [41]:
from sklearn.model_selection import StratifiedGroupKFold

# Create stratified group split
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target rate:")
print(y_train.value_counts(normalize=True))

print("\nTest target rate:")
print(y_test.value_counts(normalize=True))

print("\nShared patients:")
print(len(set(groups_train) & set(groups_test)))

Train shape: (81412, 45)
Test shape: (20354, 45)

Train target rate:
readmitted_30d
0    0.888407
1    0.111593
Name: proportion, dtype: float64

Test target rate:
readmitted_30d
0    0.888376
1    0.111624
Name: proportion, dtype: float64

Shared patients:
0


## Patient-Grouped Stratified Train/Validation Split

The training data is divided into a training subset and validation set.

The test set remains untouched until final evaluation.

The split preserves patient independence and maintains the readmission rate across the two subsets.

In [42]:
from sklearn.model_selection import StratifiedGroupKFold

# Split training data
validation_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_sub_idx, val_idx = next(
    validation_splitter.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

X_train_sub = X_train.iloc[train_sub_idx].copy()
X_val = X_train.iloc[val_idx].copy()

y_train_sub = y_train.iloc[train_sub_idx].copy()
y_val = y_train.iloc[val_idx].copy()

groups_train_sub = groups_train.iloc[train_sub_idx].copy()
groups_val = groups_train.iloc[val_idx].copy()

print("Training subset:", X_train_sub.shape)
print("Validation set:", X_val.shape)

print("\nTraining target rate:")
print(y_train_sub.value_counts(normalize=True))

print("\nValidation target rate:")
print(y_val.value_counts(normalize=True))

print("\nShared patients:")
print(len(set(groups_train_sub) & set(groups_val)))

Training subset: (65129, 45)
Validation set: (16283, 45)

Training target rate:
readmitted_30d
0    0.888406
1    0.111594
Name: proportion, dtype: float64

Validation target rate:
readmitted_30d
0    0.888411
1    0.111589
Name: proportion, dtype: float64

Shared patients:
0


In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Preprocess numeric features
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Preprocess categorical features
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Combine transformations
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
])

# Fit on training subset only
X_train_sub_processed = preprocessor.fit_transform(X_train_sub)

# Transform validation and test
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Training:", X_train_sub_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)
print("Same feature count:",
      X_train_sub_processed.shape[1]
      == X_val_processed.shape[1]
      == X_test_processed.shape[1])

Training: (65129, 2328)
Validation: (16283, 2328)
Test: (20354, 2328)
Same feature count: True


In [44]:
# Check processed data
print("Train NaN:", np.isnan(X_train_sub_processed.data).sum())
print("Validation NaN:", np.isnan(X_val_processed.data).sum())
print("Test NaN:", np.isnan(X_test_processed.data).sum())

Train NaN: 0
Validation NaN: 0
Test NaN: 0


In [45]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced",
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "HistGradient Boosting": HistGradientBoostingClassifier(
        max_iter=200,
        random_state=42
    ),
}

pipelines = {
    name: Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}

print("Models:", list(pipelines.keys()))

Models: ['Logistic Regression', 'Random Forest', 'HistGradient Boosting']


In [46]:
# Fit candidate models
for name, pipeline in pipelines.items():
    print(f"Fitting {name}...")
    pipeline.fit(X_train_sub, y_train_sub)

print("All models fitted.")

Fitting Logistic Regression...
Fitting Random Forest...
Fitting HistGradient Boosting...


TypeError: Sparse data was passed for X, but dense data is required. Use '.toarray()' to convert to a dense numpy array.

In [ ]:
# Check fitted models
for name, pipeline in pipelines.items():
    print(
        name,
        "->",
        hasattr(pipeline, "classes_")
        or hasattr(pipeline.named_steps["model"], "classes_")
    )

Logistic Regression -> True
Random Forest -> True
HistGradient Boosting -> False


In [ ]:
# Generate validation probabilities
validation_probabilities = {}

for name in ["Logistic Regression", "Random Forest"]:
    validation_probabilities[name] = pipelines[name].predict_proba(X_val)[:, 1]

print("Validation probabilities generated.")

Validation probabilities generated.


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

for name, probabilities in validation_probabilities.items():
    roc_auc = roc_auc_score(y_val, probabilities)
    pr_auc = average_precision_score(y_val, probabilities)

    print(f"{name}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  PR-AUC:  {pr_auc:.4f}")
    print()

Logistic Regression
  ROC-AUC: 0.6517
  PR-AUC:  0.2113

Random Forest
  ROC-AUC: 0.6731
  PR-AUC:  0.2262



In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

for name, probabilities in validation_probabilities.items():
    predictions = (probabilities >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions
    ).ravel()

    specificity = tn / (tn + fp)

    print(name)
    print(f"  Accuracy:          {accuracy_score(y_val, predictions):.4f}")
    print(f"  Balanced Accuracy: {balanced_accuracy_score(y_val, predictions):.4f}")
    print(f"  Precision:         {precision_score(y_val, predictions, zero_division=0):.4f}")
    print(f"  Recall:            {recall_score(y_val, predictions):.4f}")
    print(f"  F1:                {f1_score(y_val, predictions):.4f}")
    print(f"  Specificity:       {specificity:.4f}")
    print(f"  TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
    print()

Logistic Regression
  Accuracy:          0.6557
  Balanced Accuracy: 0.6097
  Precision:         0.1727
  Recall:            0.5504
  F1:                0.2630
  Specificity:       0.6689
  TN: 9677, FP: 4789, FN: 817, TP: 1000

Random Forest
  Accuracy:          0.8887
  Balanced Accuracy: 0.5122
  Precision:         0.5208
  Recall:            0.0275
  F1:                0.0523
  Specificity:       0.9968
  TN: 14420, FP: 46, FN: 1767, TP: 50



In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
)

thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for name, probabilities in validation_probabilities.items():
    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)

        threshold_results.append({
            "model": name,
            "threshold": threshold,
            "precision": precision_score(
                y_val, predictions, zero_division=0
            ),
            "recall": recall_score(
                y_val, predictions, zero_division=0
            ),
            "f1": f1_score(
                y_val, predictions, zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_val, predictions
            ),
        })

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,model,threshold,precision,recall,f1,balanced_accuracy
0,Logistic Regression,0.10,0.114767,0.994496,0.205785,0.515498
1,Logistic Regression,0.15,0.116245,0.990094,0.208061,0.522318
2,Logistic Regression,0.20,0.117565,0.973583,0.209796,0.527853
3,Logistic Regression,0.25,0.121206,0.949367,0.214967,0.542394
4,Logistic Regression,0.30,0.126289,0.903137,0.221592,0.559166
5,Logistic Regression,0.35,0.132293,0.821134,0.227873,0.572325
6,Logistic Regression,0.40,0.142374,0.740782,0.238843,0.590147
7,Logistic Regression,0.45,0.155120,0.644469,0.250053,0.601787
8,Logistic Regression,0.50,0.172741,0.550358,0.262950,0.609653
9,Logistic Regression,0.55,0.191914,0.454595,0.269891,0.607085


In [ ]:
for name, probabilities in validation_probabilities.items():
    print(name)
    print(pd.Series(probabilities).describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ))
    print()

Logistic Regression
count    16283.000000
mean         0.442250
std          0.178812
min          0.000471
1%           0.006742
5%           0.151083
10%          0.228251
25%          0.324316
50%          0.434024
75%          0.558052
90%          0.681691
95%          0.754004
99%          0.864868
max          0.986816
dtype: float64

Random Forest
count    16283.000000
mean         0.207881
std          0.092970
min          0.000000
1%           0.046667
5%           0.076667
10%          0.100000
25%          0.140000
50%          0.196667
75%          0.263333
90%          0.333333
95%          0.376667
99%          0.473333
max          0.703333
dtype: float64



In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

rf_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 5, 10],
}

print("CV folds:", rf_cv.get_n_splits())
print("Hyperparameter combinations:", 27)
print(rf_param_grid)

CV folds: 5
Hyperparameter combinations: 27
{'model__n_estimators': [200, 300, 500], 'model__max_depth': [None, 10, 20], 'model__min_samples_leaf': [1, 5, 10]}


In [ ]:
from sklearn.model_selection import GridSearchCV

rf_search = GridSearchCV(
    estimator=pipelines["Random Forest"],
    param_grid=rf_param_grid,
    scoring="average_precision",
    cv=rf_cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

rf_search.fit(
    X_train_sub,
    y_train_sub,
    groups=groups_train_sub,
)

print("Best PR-AUC:", rf_search.best_score_)
print("Best parameters:", rf_search.best_params_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


c:\Users\pc\diabetes-readmission-ml\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best PR-AUC: 0.22007491728676615
Best parameters: {'model__max_depth': None, 'model__min_samples_leaf': 5, 'model__n_estimators': 500}


In [ ]:
tuned_rf_probabilities = rf_search.predict_proba(X_val)[:, 1]

tuned_rf_roc_auc = roc_auc_score(
    y_val,
    tuned_rf_probabilities
)

tuned_rf_pr_auc = average_precision_score(
    y_val,
    tuned_rf_probabilities
)

print("Tuned Random Forest")
print(f"  ROC-AUC: {tuned_rf_roc_auc:.4f}")
print(f"  PR-AUC:  {tuned_rf_pr_auc:.4f}")

Tuned Random Forest
  ROC-AUC: 0.6805
  PR-AUC:  0.2240


In [ ]:
tuned_threshold_results = []

for threshold in thresholds:
    predictions = (
        tuned_rf_probabilities >= threshold
    ).astype(int)

    tuned_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, predictions
        ),
    })

tuned_threshold_results = pd.DataFrame(
    tuned_threshold_results
)

tuned_threshold_results

,threshold,precision,recall,f1,balanced_accuracy
0,0.10,0.111589,1.000000,0.200773,0.500000
1,0.15,0.111630,1.000000,0.200840,0.500207
2,0.20,0.111884,1.000000,0.201252,0.501486
3,0.25,0.112739,0.999450,0.202622,0.505739
4,0.30,0.115275,0.997248,0.206661,0.517945
5,0.35,0.122722,0.959824,0.217619,0.549005
6,0.40,0.136328,0.871767,0.235784,0.589036
7,0.45,0.158713,0.730325,0.260758,0.622041
8,0.50,0.190208,0.538800,0.281160,0.625338
9,0.55,0.240969,0.323060,0.276040,0.597622


In [ ]:
test_probabilities = rf_search.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= 0.50
).astype(int)

test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

test_pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

test_precision = precision_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_balanced_accuracy = balanced_accuracy_score(
    y_test,
    test_predictions
)

print("Final Test Evaluation")
print(f"ROC-AUC:           {test_roc_auc:.4f}")
print(f"PR-AUC:            {test_pr_auc:.4f}")
print(f"Precision:         {test_precision:.4f}")
print(f"Recall:            {test_recall:.4f}")
print(f"F1:                {test_f1:.4f}")
print(f"Balanced Accuracy: {test_balanced_accuracy:.4f}")

Final Test Evaluation
ROC-AUC:           0.6770
PR-AUC:            0.2088
Precision:         0.1868
Recall:            0.5299
F1:                0.2762
Balanced Accuracy: 0.6200


In [ ]:
test_predictions = (test_probabilities >= 0.50).astype(int)

error_analysis = X_test.copy()
error_analysis["actual"] = y_test.values
error_analysis["predicted"] = test_predictions
error_analysis["probability"] = test_probabilities

error_analysis["error_type"] = "True Negative"
error_analysis.loc[
    (error_analysis["actual"] == 0) & (error_analysis["predicted"] == 1),
    "error_type"
] = "False Positive"
error_analysis.loc[
    (error_analysis["actual"] == 1) & (error_analysis["predicted"] == 0),
    "error_type"
] = "False Negative"
error_analysis.loc[
    (error_analysis["actual"] == 1) & (error_analysis["predicted"] == 1),
    "error_type"
] = "True Positive"

error_analysis["error_type"].value_counts()

error_type
True Negative     12840
False Positive     5242
True Positive      1204
False Negative     1068
Name: count, dtype: int64

In [ ]:
# Summarize error groups
error_summary = (
    error_analysis.groupby("error_type")
    .agg(
        count=("actual", "size"),
        mean_probability=("probability", "mean"),
        mean_time_in_hospital=("time_in_hospital", "mean"),
        mean_num_medications=("num_medications", "mean"),
        mean_number_inpatient=("number_inpatient", "mean"),
        mean_number_diagnoses=("number_diagnoses", "mean"),
    )
    .sort_values("count", ascending=False)
)

error_summary

,count,mean_probability,mean_time_in_hospital,mean_num_medications,mean_number_inpatient,mean_number_diagnoses
error_type,,,,,,
True Negative,12840,0.405852,3.891978,15.067757,0.217134,7.110280
False Positive,5242,0.552112,5.509348,18.025944,1.429989,8.181992
True Positive,1204,0.568210,5.269934,18.045681,1.955980,8.029070
False Negative,1068,0.431112,4.249064,15.809925,0.328652,7.490637


In [ ]:
# Compare false positives and true positives
fp_tp = error_analysis[
    error_analysis["error_type"].isin(["False Positive", "True Positive"])
].copy()

fp_tp.groupby("error_type")[
    [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
].mean().T

error_type,False Positive,True Positive
time_in_hospital,5.509348,5.269934
num_lab_procedures,46.538153,46.180233
num_procedures,1.129531,1.108804
num_medications,18.025944,18.045681
number_outpatient,0.624762,0.646179
number_emergency,0.391072,0.539867
number_inpatient,1.429989,1.955980
number_diagnoses,8.181992,8.029070


In [ ]:
# Compare false negatives and true positives
fn_tp = error_analysis[
    error_analysis["error_type"].isin(["False Negative", "True Positive"])
].copy()

fn_tp.groupby("error_type")[
    [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
].mean().T

error_type,False Negative,True Positive
time_in_hospital,4.249064,5.269934
num_lab_procedures,42.904494,46.180233
num_procedures,1.510300,1.108804
num_medications,15.809925,18.045681
number_outpatient,0.346442,0.646179
number_emergency,0.106742,0.539867
number_inpatient,0.328652,1.955980
number_diagnoses,7.490637,8.029070


In [ ]:
# Compare false negatives and true positives by key categories
categorical_error_features = [
    "age",
    "discharge_disposition",
]

for feature in categorical_error_features:
    print(f"\n{feature}")
    print(
        pd.crosstab(
            error_analysis[feature],
            error_analysis["error_type"],
            normalize="columns"
        ).round(3)
    )


age
error_type  False Negative  False Positive  True Negative  True Positive
age                                                                     
[0-10)               0.001           0.000          0.002          0.000
[10-20)              0.002           0.002          0.008          0.000
[20-30)              0.006           0.016          0.014          0.022
[30-40)              0.042           0.028          0.040          0.037
[40-50)              0.078           0.074          0.104          0.090
[50-60)              0.169           0.119          0.192          0.135
[60-70)              0.227           0.204          0.230          0.223
[70-80)              0.275           0.285          0.242          0.274
[80-90)              0.170           0.238          0.145          0.195
[90-100)             0.031           0.035          0.023          0.023

discharge_disposition
error_type                                          False Negative  \
discharge_disposition     

In [ ]:
# Calculate readmission rates by discharge disposition
disposition_rates = (
    data.groupby("discharge_disposition")["readmitted_30d"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

disposition_rates

,count,mean
discharge_disposition,,
Still patient or expected to return for outpatient services,3,0.666667
Discharged/transferred within this institution to Medicare approved swing bed,63,0.444444
Admitted as an inpatient to this hospital,21,0.428571
Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital,139,0.366906
Discharged/transferred to another rehab fac including rehab units of a hospital .,1993,0.276969
Discharged/transferred to another type of inpatient care institution,1184,0.208615
Discharged/transferred to another short term hospital,2128,0.160714
Discharged/transferred to SNF,13954,0.146625
Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.,48,0.145833


In [ ]:
# Calculate error rates within each discharge group
major_dispositions = [
    "Discharged to home",
    "Discharged/transferred to SNF",
    "Discharged/transferred to home with home health service",
]

disposition_errors = (
    error_analysis[
        error_analysis["discharge_disposition"].isin(major_dispositions)
    ]
    .groupby("discharge_disposition")["error_type"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)

disposition_errors.round(3)

error_type,False Negative,False Positive,True Negative,True Positive
discharge_disposition,,,,
Discharged to home,0.061,0.133,0.776,0.030
Discharged/transferred to SNF,0.032,0.587,0.268,0.114
Discharged/transferred to home with home health service,0.059,0.364,0.501,0.075


In [ ]:
# Extract feature importance from the final Random Forest
final_rf = rf_search.best_estimator_

feature_names = final_rf.named_steps["preprocessor"].get_feature_names_out()
importances = final_rf.named_steps["model"].feature_importances_

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances,
    })
    .sort_values("importance", ascending=False)
    .head(20)
)

feature_importance

,feature,importance
6,numeric__number_inpatient,0.119391
3,numeric__num_medications,0.035825
2303,categorical__discharge_disposition_Discharged ...,0.032562
1,numeric__num_lab_procedures,0.032308
2318,categorical__discharge_disposition_Expired,0.030466
0,numeric__time_in_hospital,0.029363
5,numeric__number_emergency,0.028212
7,numeric__number_diagnoses,0.024304
2309,categorical__discharge_disposition_Discharged/...,0.023963
2,numeric__num_procedures,0.016699


In [ ]:
# Show full names of the top features
pd.set_option("display.max_colwidth", None)

feature_importance.head(20)

,feature,importance
6,numeric__number_inpatient,0.119391
3,numeric__num_medications,0.035825
2303,categorical__discharge_disposition_Discharged to home,0.032562
1,numeric__num_lab_procedures,0.032308
2318,categorical__discharge_disposition_Expired,0.030466
0,numeric__time_in_hospital,0.029363
5,numeric__number_emergency,0.028212
7,numeric__number_diagnoses,0.024304
2309,categorical__discharge_disposition_Discharged/transferred to another rehab fac including rehab units of a hospital .,0.023963
2,numeric__num_procedures,0.016699


In [ ]:
# Check actual readmission rate across prediction-risk bands
risk_bands = pd.qcut(
    error_analysis["probability"],
    q=10,
    duplicates="drop"
)

risk_band_summary = (
    error_analysis.groupby(risk_bands, observed=True)
    .agg(
        encounters=("actual", "size"),
        actual_readmission_rate=("actual", "mean"),
        mean_predicted_probability=("probability", "mean"),
    )
)

risk_band_summary

,encounters,actual_readmission_rate,mean_predicted_probability
probability,,,
"(0.128, 0.34]",2036,0.034381,0.303751
"(0.34, 0.376]",2035,0.052088,0.359080
"(0.376, 0.404]",2035,0.057985,0.389927
"(0.404, 0.43]",2036,0.080550,0.416890
"(0.43, 0.455]",2035,0.087469,0.442365
"(0.455, 0.479]",2035,0.102703,0.466944
"(0.479, 0.504]",2036,0.127701,0.491350
"(0.504, 0.533]",2035,0.135627,0.517923
"(0.533, 0.57]",2035,0.177887,0.550770


In [ ]:
# Rebuild the final model with frozen settings
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ))
])

final_model.fit(X_train_sub, y_train_sub)

print("Final model fitted.")

Final model fitted.


In [ ]:
# Import evaluation metrics
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Verify the reconstructed model on the held-out test set
final_test_probabilities = final_model.predict_proba(X_test)[:, 1]
final_test_predictions = (final_test_probabilities >= 0.50).astype(int)

print("Test ROC-AUC:", roc_auc_score(y_test, final_test_probabilities))
print("Test PR-AUC:", average_precision_score(y_test, final_test_probabilities))
print("Test F1:", f1_score(y_test, final_test_predictions))
print("Test recall:", recall_score(y_test, final_test_predictions))
print("Test precision:", precision_score(y_test, final_test_predictions))

Test ROC-AUC: 0.6769744462238534
Test PR-AUC: 0.20879884514572353
Test F1: 0.2762101399403533
Test recall: 0.5299295774647887
Test precision: 0.18678250077567485


In [ ]:
# Create the model output directory
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

print("Models directory:", models_dir.resolve())

Models directory: C:\Users\pc\diabetes-readmission-ml\models


In [ ]:
# Save the verified final model
import joblib

joblib.dump(
    final_model,
    "../models/final_random_forest.joblib"
)

print("Final model saved.")

Final model saved.


In [ ]:
# Verify the saved model artifact directly
from pathlib import Path

model_path = Path("../models/final_random_forest.joblib")

print("Exists:", model_path.exists())
print("Size (bytes):", model_path.stat().st_size)
print("Path:", model_path.resolve())

Exists: True
Size (bytes): 74472171
Path: C:\Users\pc\diabetes-readmission-ml\models\final_random_forest.joblib


In [ ]:
# Verify the saved model can be loaded
import joblib

loaded_model = joblib.load("../models/final_random_forest.joblib")

print("Model type:", type(loaded_model).__name__)
print("Pipeline steps:", list(loaded_model.named_steps.keys()))

Model type: Pipeline
Pipeline steps: ['preprocessor', 'model']


In [ ]:
# Calculate validation permutation importance
from sklearn.inspection import permutation_importance

permutation_result = permutation_importance(
    final_model,
    X_val,
    y_val,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)

permutation_importance_df = (
    pd.DataFrame({
        "feature": X_val.columns,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

permutation_importance_df

,feature,importance_mean,importance_std
44,discharge_disposition,0.040858,0.002755
6,number_inpatient,0.038452,0.003690
13,diag_1,0.012071,0.001584
14,diag_2,0.003562,0.001034
43,medical_specialty,0.003529,0.000844
15,diag_3,0.003506,0.002104
5,number_emergency,0.003275,0.000646
11,admission_type,0.002179,0.000512
17,A1Cresult,0.001881,0.000670
10,age,0.001656,0.000704


In [ ]:
# Summarize cumulative permutation importance
permutation_ranked = permutation_importance_df.copy()

permutation_ranked["cumulative_importance"] = (
    permutation_ranked["importance_mean"].clip(lower=0).cumsum()
)

permutation_ranked[
    ["feature", "importance_mean", "importance_std", "cumulative_importance"]
].head(20)

,feature,importance_mean,importance_std,cumulative_importance
44,discharge_disposition,0.040858,0.002755,0.040858
6,number_inpatient,0.038452,0.003690,0.079310
13,diag_1,0.012071,0.001584,0.091381
14,diag_2,0.003562,0.001034,0.094942
43,medical_specialty,0.003529,0.000844,0.098472
15,diag_3,0.003506,0.002104,0.101978
5,number_emergency,0.003275,0.000646,0.105253
11,admission_type,0.002179,0.000512,0.107432
17,A1Cresult,0.001881,0.000670,0.109314
10,age,0.001656,0.000704,0.110970


In [ ]:
# Select the top permutation-ranked candidate features
sfs_candidates = permutation_importance_df.head(20)["feature"].tolist()

print("Candidate feature count:", len(sfs_candidates))
print("Candidate features:")
print(sfs_candidates)

Candidate feature count: 20
Candidate features:
['discharge_disposition', 'number_inpatient', 'diag_1', 'diag_2', 'medical_specialty', 'diag_3', 'number_emergency', 'admission_type', 'A1Cresult', 'age', 'num_procedures', 'number_outpatient', 'num_lab_procedures', 'num_medications', 'admission_source', 'time_in_hospital', 'gender', 'max_glu_serum', 'number_diagnoses', 'glimepiride']


In [ ]:
# Run sequential forward selection with grouped cross-validation
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold

sfs_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

numeric_set = set(numeric_features)
categorical_set = set(categorical_features)


def build_feature_pipeline(features):
    selected_numeric = [
        feature for feature in features
        if feature in numeric_set
    ]

    selected_categorical = [
        feature for feature in features
        if feature in categorical_set
    ]

    selected_preprocessor = ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            selected_numeric,
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="Missing",
                    ),
                ),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore"),
                ),
            ]),
            selected_categorical,
        ),
    ])

    return Pipeline([
        ("preprocessor", selected_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=None,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ])


selected_features = []
remaining_features = sfs_candidates.copy()
sfs_results = []

for step in range(10):
    best_feature = None
    best_score = -float("inf")

    for candidate in remaining_features:
        current_features = selected_features + [candidate]
        fold_scores = []

        for train_fold, val_fold in sfs_cv.split(
            X_train_sub,
            y_train_sub,
            groups_train_sub,
        ):
            fold_model = build_feature_pipeline(current_features)

            fold_model.fit(
                X_train_sub.iloc[train_fold][current_features],
                y_train_sub.iloc[train_fold],
            )

            fold_probabilities = fold_model.predict_proba(
                X_train_sub.iloc[val_fold][current_features]
            )[:, 1]

            fold_score = average_precision_score(
                y_train_sub.iloc[val_fold],
                fold_probabilities,
            )

            fold_scores.append(fold_score)

        mean_score = sum(fold_scores) / len(fold_scores)

        if mean_score > best_score:
            best_score = mean_score
            best_feature = candidate

    selected_features.append(best_feature)
    remaining_features.remove(best_feature)

    sfs_results.append({
        "step": step + 1,
        "feature_added": best_feature,
        "cv_pr_auc": best_score,
    })

    print(
        f"Step {step + 1}: "
        f"{best_feature} | "
        f"CV PR-AUC: {best_score:.4f}"
    )

print("\nSelected features:")
print(selected_features)

sfs_results = pd.DataFrame(sfs_results)

Step 1: number_inpatient | CV PR-AUC: 0.1740
Step 2: discharge_disposition | CV PR-AUC: 0.2053
Step 3: medical_specialty | CV PR-AUC: 0.2102
Step 4: admission_source | CV PR-AUC: 0.2165
Step 5: number_emergency | CV PR-AUC: 0.2192


c:\Users\pc\diabetes-readmission-ml\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\pc\diabetes-readmission-ml\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\pc\diabetes-readmission-ml\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\pc\diabetes-readmission-ml\.venv\Lib\sit

Step 6: admission_type | CV PR-AUC: 0.2216
Step 7: num_procedures | CV PR-AUC: 0.2224
Step 8: number_diagnoses | CV PR-AUC: 0.2237
Step 9: max_glu_serum | CV PR-AUC: 0.2243
Step 10: A1Cresult | CV PR-AUC: 0.2234

Selected features:
['number_inpatient', 'discharge_disposition', 'medical_specialty', 'admission_source', 'number_emergency', 'admission_type', 'num_procedures', 'number_diagnoses', 'max_glu_serum', 'A1Cresult']


In [ ]:
# Select the best SFS subset
best_sfs_step = sfs_results.loc[
    sfs_results["cv_pr_auc"].idxmax()
]

best_sfs_features = sfs_results.loc[
    :best_sfs_step["step"] - 1,
    "feature_added"
].tolist()

print("Best SFS step:", int(best_sfs_step["step"]))
print("Best CV PR-AUC:", best_sfs_step["cv_pr_auc"])
print("Selected feature count:", len(best_sfs_features))
print("Selected features:")
print(best_sfs_features)

Best SFS step: 9
Best CV PR-AUC: 0.2242964622989548
Selected feature count: 9
Selected features:
['number_inpatient', 'discharge_disposition', 'medical_specialty', 'admission_source', 'number_emergency', 'admission_type', 'num_procedures', 'number_diagnoses', 'max_glu_serum']


In [ ]:
selected_feature_model = build_feature_pipeline(best_sfs_features)

selected_feature_model.set_params(
    model__n_estimators=500
)

selected_feature_model.fit(
    X_train_sub[best_sfs_features],
    y_train_sub
)

selected_val_probabilities = selected_feature_model.predict_proba(
    X_val[best_sfs_features]
)[:, 1]

selected_val_predictions = (
    selected_val_probabilities >= 0.50
).astype(int)

print("Selected-feature model fitted.")
print("Number of features:", len(best_sfs_features))
print("Processed feature count:",
      selected_feature_model.named_steps["preprocessor"].transform(
          X_train_sub[best_sfs_features]
      ).shape[1])

Selected-feature model fitted.
Number of features: 9
Processed feature count: 126


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
)

selected_val_metrics = {
    "ROC-AUC": roc_auc_score(y_val, selected_val_probabilities),
    "PR-AUC": average_precision_score(y_val, selected_val_probabilities),
    "Precision": precision_score(y_val, selected_val_predictions),
    "Recall": recall_score(y_val, selected_val_predictions),
    "F1": f1_score(y_val, selected_val_predictions),
    "Balanced Accuracy": balanced_accuracy_score(
        y_val,
        selected_val_predictions
    ),
}

print("9-feature validation metrics:")
for metric, value in selected_val_metrics.items():
    print(f"{metric}: {value:.4f}")

9-feature validation metrics:
ROC-AUC: 0.6708
PR-AUC: 0.2278
Precision: 0.1746
Recall: 0.5993
F1: 0.2704
Balanced Accuracy: 0.6217


In [ ]:
selected_test_probabilities = selected_feature_model.predict_proba(
    X_test[best_sfs_features]
)[:, 1]

selected_test_predictions = (
    selected_test_probabilities >= 0.50
).astype(int)

selected_test_metrics = {
    "ROC-AUC": roc_auc_score(y_test, selected_test_probabilities),
    "PR-AUC": average_precision_score(
        y_test,
        selected_test_probabilities
    ),
    "Precision": precision_score(
        y_test,
        selected_test_predictions
    ),
    "Recall": recall_score(
        y_test,
        selected_test_predictions
    ),
    "F1": f1_score(
        y_test,
        selected_test_predictions
    ),
    "Balanced Accuracy": balanced_accuracy_score(
        y_test,
        selected_test_predictions
    ),
}

print("9-feature test metrics:")
for metric, value in selected_test_metrics.items():
    print(f"{metric}: {value:.4f}")

9-feature test metrics:
ROC-AUC: 0.6726
PR-AUC: 0.2198
Precision: 0.1763
Recall: 0.6193
F1: 0.2745
Balanced Accuracy: 0.6279


In [51]:
import joblib
from pathlib import Path

model_path = Path("../models/final_random_forest_9_features.joblib")

joblib.dump(
    selected_feature_model,
    model_path
)

print("Saved:", model_path.resolve())
print("Exists:", model_path.exists())
print("Size (bytes):", model_path.stat().st_size)

Saved: C:\Users\pc\diabetes-readmission-ml\models\final_random_forest_9_features.joblib
Exists: True
Size (bytes): 54700539


In [ ]:
import joblib

loaded_selected_model = joblib.load(
    "../models/final_random_forest_9_features.joblib"
)

print("Loaded model type:", type(loaded_selected_model).__name__)
print("Pipeline steps:", list(loaded_selected_model.named_steps.keys()))

loaded_test_probabilities = loaded_selected_model.predict_proba(
    X_test[best_sfs_features]
)[:, 1]

print(
    "Predictions match:",
    (loaded_test_probabilities == selected_test_probabilities).all()
)

Loaded model type: Pipeline
Pipeline steps: ['preprocessor', 'model']
Predictions match: False


In [ ]:
import numpy as np

difference = np.abs(
    loaded_test_probabilities - selected_test_probabilities
)

print("Maximum absolute difference:", difference.max())
print("Mean absolute difference:", difference.mean())
print(
    "Exact matches:",
    np.sum(loaded_test_probabilities == selected_test_probabilities),
    "/",
    len(selected_test_probabilities)
)
print(
    "Close within 1e-12:",
    np.allclose(
        loaded_test_probabilities,
        selected_test_probabilities,
        rtol=1e-12,
        atol=1e-12
    )
)

Maximum absolute difference: 4.440892098500626e-16
Mean absolute difference: 3.7519595479135196e-17
Exact matches: 11230 / 20354
Close within 1e-12: True


In [ ]:
final_model_spec = {
    "model": "Random Forest",
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_leaf": 5,
    "class_weight": "balanced",
    "random_state": 42,
    "threshold": 0.50,
    "raw_feature_count": len(best_sfs_features),
    "processed_feature_count": 126,
    "features": best_sfs_features,
}

for key, value in final_model_spec.items():
    print(f"{key}: {value}")

model: Random Forest
n_estimators: 500
max_depth: None
min_samples_leaf: 5
class_weight: balanced
random_state: 42
threshold: 0.5
raw_feature_count: 9
processed_feature_count: 126
features: ['number_inpatient', 'discharge_disposition', 'medical_specialty', 'admission_source', 'number_emergency', 'admission_type', 'num_procedures', 'number_diagnoses', 'max_glu_serum']
